#Data Acquisition Casestudy Answers

##Step 1: Load SpaceX Launch Data

In [28]:
import pandas as pd
import requests

# getting launch data from SpaceX API
url1 = "https://api.spacexdata.com/v4/launches"
data1 = requests.get(url1).json()

# converting to dataframe
df1 = pd.DataFrame(data1)

# selecting only needed columns
df1 = df1[['name','date_utc','success','details','rocket']]

# converting date and extracting year
df1['date_utc'] = pd.to_datetime(df1['date_utc'])
df1['year'] = df1['date_utc'].dt.year

df1.head()


,name,date_utc,success,details,rocket,year
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009


##Step 2: Load Rocket Metadata

In [29]:
# getting rocket details
url2 = "https://api.spacexdata.com/v4/rockets"
data2 = requests.get(url2).json()

df2 = pd.DataFrame(data2)

# selecting required columns
df2 = df2[['id','name','type','active','stages']]

df2.head()


,id,name,type,active,stages
0,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
1,5e9d0d95eda69973a809d1ec,Falcon 9,rocket,True,2
2,5e9d0d95eda69974db09d1ed,Falcon Heavy,rocket,True,2
3,5e9d0d96eda699382d09d1ee,Starship,rocket,False,2


Step 3: Merge Data

In [30]:
# merging using rocket id
df = pd.merge(df1, df2, left_on='rocket', right_on='id', how='left')

df.head()


,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2


##Step 4: Add Simulated Country

In [31]:
import random

countries = ['USA','Russia','India','China','France']

# assigning random country
df['country'] = [random.choice(countries) for i in range(len(df))]

df.head()


,name_x,date_utc,success,details,rocket,year,id,name_y,type,active,stages,country
0,FalconSat,2006-03-24 22:30:00+00:00,False,Engine failure at 33 seconds and loss of vehicle,5e9d0d95eda69955f709d1eb,2006,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,USA
1,DemoSat,2007-03-21 01:10:00+00:00,False,Successful first stage burn and transition to ...,5e9d0d95eda69955f709d1eb,2007,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,Russia
2,Trailblazer,2008-08-03 03:34:00+00:00,False,Residual stage 1 thrust led to collision betwe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,Russia
3,RatSat,2008-09-28 23:15:00+00:00,True,Ratsat was carried to orbit on the first succe...,5e9d0d95eda69955f709d1eb,2008,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,France
4,RazakSat,2009-07-13 03:35:00+00:00,True,None,5e9d0d95eda69955f709d1eb,2009,5e9d0d95eda69955f709d1eb,Falcon 1,rocket,False,2,France


##Step 5: Store Data in SQLite

In [32]:
import sqlite3

# creating database
conn = sqlite3.connect("spacex.db")

# saving table
df.to_sql("launches", conn, if_exists='replace', index=False)

print("Table created successfully")


Table created successfully


##Step 6: SQL Queries

###1. Launches by Country

In [33]:
q1 = """
SELECT country, COUNT(*) as total
FROM launches
GROUP BY country
ORDER BY total DESC;
"""
pd.read_sql(q1, conn)


,country,total
0,USA,46
1,India,43
2,Russia,39
3,France,39
4,China,38


###2. Year with Highest Launches

In [34]:
q2 = """
SELECT year, COUNT(*) as total
FROM launches
GROUP BY year
ORDER BY total DESC
LIMIT 1;
"""
pd.read_sql(q2, conn)


,year,total
0,2022,62


###3. Top 5 Missions

In [35]:
df.columns


Index(['name_x', 'date_utc', 'success', 'details', 'rocket', 'year', 'id',
       'name_y', 'type', 'active', 'stages', 'country'],
      dtype='object')

In [36]:
q3 = """
SELECT name_x, COUNT(*) as total
FROM launches
GROUP BY name_x
ORDER BY total DESC
LIMIT 5;
"""
pd.read_sql(q3, conn)


,name_x,total
0,ispace Mission 1 & Rashid,1
1,ZUMA,1
2,WorldView Legion 1 & 2,1
3,Viasat-3 & Arcturus,1
4,USSF-44,1
